<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/riesgo/notebooks/c4_l2.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C4-L2 · SL/TP OCO
Riesgo, beneficio y RR fila por fila con 50 trades OCO. Winrate, breakeven y expectancy.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/riesgo/data/c4_l2.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c4_l2.csv'), Path('data/c4_l2.csv'), Path('c4_l2.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)

In [ ]:
df['riesgo_calc'] = (df['entrada'] - df['stop']).abs()
df['beneficio_calc'] = (df['take'] - df['entrada']).abs()
df['rr_calc'] = df['beneficio_calc'] / df['riesgo_calc']
print(df[['trade','riesgo','riesgo_calc','beneficio','beneficio_calc','rr','rr_calc']].head(8).to_string(index=False))
print(f"RR medio={df['rr'].mean():.3f}  min={df['rr'].min():.3f}")

In [ ]:
winrate = (df['resultado'] == 'win').mean()
breakeven = 1 / (1 + df['rr'].mean())
exp = df.groupby('resultado')['pnl'].mean()
expectancy = winrate * exp.get('win', 0) + (1 - winrate) * exp.get('loss', 0)
print(f'winrate={winrate:.1%}  breakeven={breakeven:.1%}  expectancy={expectancy:+.2f} USD/trade')

In [ ]:
# Chequeo automático
assert ((df['rr'] - df['rr_calc']).abs().max() < 1e-3), 'RR no replica'
assert 1.8 < df['rr'].mean() < 2.2, 'el RR medio debe rondar 2'
assert set(df['resultado'].unique()) <= {'win', 'loss'}
print('OK: SL/TP OCO verificado')